Movielens Latest Small: with 100,000 ratings and  3,600 tag applications applied to 9,000 movies by 600 users. Last updated 9/2018.

Original data taken from: https://www.kaggle.com/datasets/grouplens/movielens-latest-small.

This data will be used for additional recommendation, using user profiles. Just for experimentation.

The movielens-clean.csv file already contains cleanded version of Movielens Latest Small, which:
 - remove all duplicated movieId
 - remove all movies with genre: imax, children, film-noir, western
 - remove all movies before year 2000
 - change genre "musical" to "music"
 - convert genres to lowercase
 - split title column to separated "name" and "year" column
 - change "movieId" to "id"

`movielens-extended`: schema compatible with local database, ready for recsys pipeline

In [9]:
import numpy as np
import pandas as pd
import sqlite3

In [10]:
con = sqlite3.connect('../data/movies.db')
con.execute('PRAGMA foreign_keys = ON')  # enable foreign keys constraint and ON DELETE CASCADE

personal_df = pd.read_sql_query("SELECT * FROM movie_detail", con)
personal_df = personal_df.drop(columns='note')
personal_df.tail()

,id,name,year,status,type,country,genres,rating,watched_date
357,358,Hoppers,2026,waiting,movie,US,"comedy,animation,sci-fi",NaN,None
358,359,Little Amélie or the Character of Rain,2025,waiting,movie,None,"animation,family,drama",NaN,None
359,360,Once We Were Us,2025,waiting,movie,Korea,"romance,drama",NaN,None
360,361,Snowpiercer,2013,waiting,movie,Korea,"action,sci-fi,thriller",NaN,None
361,362,Lời Nguyền Thư Viện Mikura (Whoever Steals Thi...,2025,waiting,movie,Japan,"animation,fantasy",NaN,None


In [11]:
personal_df.shape

(362, 9)

In [12]:
personal_unwatched_ids = personal_df[personal_df['status'] == 'waiting'].index
# personal_completed_ids = personal_df[personal_df['status] == 'completed']['id']
watched_df = personal_df[personal_df['status'] == 'completed']
personal_df = personal_df[personal_df['status'] != 'dropped']  # remove dropped movies
personal_df.shape

(358, 9)

In [46]:
personal_unwatched_ids[:5]

Index([55, 56, 57, 58, 59], dtype='int64')

In [21]:
movielens_df = pd.read_csv('../data/movielens-extended.csv')
movielens_df.head()

,id,name,year,status,type,country,genres,rating,watched_date
0,3273,Scream 3,2000,waiting,movie,NaN,"comedy,horror,mystery,thriller",NaN,NaN
1,3578,Gladiator,2000,waiting,movie,NaN,"action,adventure,drama",NaN,NaN
2,3617,Road Trip,2000,waiting,movie,NaN,comedy,NaN,NaN
3,3744,Shaft,2000,waiting,movie,NaN,"action,crime,thriller",NaN,NaN
4,3793,X-Men,2000,waiting,movie,NaN,"action,adventure,sci-fi",NaN,NaN


In [22]:
movielens_df.shape

(4412, 9)

In [26]:
all(movielens_df.columns == personal_df.columns)

True

In [27]:
# get all duplicated movie with year:
personal_df['name'] = personal_df['name'].str.strip()
movielens_df['name'] = movielens_df['name'].str.strip()

overlap = pd.merge(
    personal_df[['name', 'year']],
    movielens_df[['name', 'year']],
    on=['name', 'year'],
    how='inner',
    suffixes=('_personal', '_movielens')
)
overlap

,name,year
0,You Are the Apple of My Eye,2011
1,Blade Runner 2049,2017
2,10 Cloverfield Lane,2016
3,Inception,2010
4,The Boy and the Beast,2015
5,Her,2013
6,Whiplash,2014
7,The Martian,2015
8,Lady Bird,2017
9,3 Idiots,2009


In [28]:
# remove overlap movies from movielens_df
overlap_keys = set(zip(overlap['name'], overlap['year']))
# Create a boolean mask - True for rows NOT in overlap
mask = movielens_df.apply(
    lambda row: (row['name'], row['year']) not in overlap_keys, axis=1
)
movielens_df = movielens_df[mask].reset_index(drop=True)
movielens_df.shape

(4363, 9)

In [30]:
movielens_df.head()

,id,name,year,status,type,country,genres,rating,watched_date
0,3273,Scream 3,2000,waiting,movie,NaN,"comedy,horror,mystery,thriller",NaN,NaN
1,3578,Gladiator,2000,waiting,movie,NaN,"action,adventure,drama",NaN,NaN
2,3617,Road Trip,2000,waiting,movie,NaN,comedy,NaN,NaN
3,3744,Shaft,2000,waiting,movie,NaN,"action,crime,thriller",NaN,NaN
4,3793,X-Men,2000,waiting,movie,NaN,"action,adventure,sci-fi",NaN,NaN


### Merging data

In [35]:
personal_df.shape

(358, 9)

In [32]:
# the movielens data's smallest id is 2769 and largest is 193587 -> spare -> reindex
start_id = personal_df['id'].max() + 1
movielens_df['id'] = range(start_id, start_id + len(movielens_df))
movielens_df.head()

,id,name,year,status,type,country,genres,rating,watched_date
0,363,Scream 3,2000,waiting,movie,NaN,"comedy,horror,mystery,thriller",NaN,NaN
1,364,Gladiator,2000,waiting,movie,NaN,"action,adventure,drama",NaN,NaN
2,365,Road Trip,2000,waiting,movie,NaN,comedy,NaN,NaN
3,366,Shaft,2000,waiting,movie,NaN,"action,crime,thriller",NaN,NaN
4,367,X-Men,2000,waiting,movie,NaN,"action,adventure,sci-fi",NaN,NaN


In [36]:
# ensure uniqueness before merging
set(personal_df['id']).intersection(set(movielens_df['id']))

set()

In [50]:
# testing merging
df = pd.concat([personal_df, movielens_df], ignore_index=True)
df.iloc[356:362]

,id,name,year,status,type,country,genres,rating,watched_date
356,361,Snowpiercer,2013,waiting,movie,Korea,"action,sci-fi,thriller",NaN,None
357,362,Lời Nguyền Thư Viện Mikura (Whoever Steals Thi...,2025,waiting,movie,Japan,"animation,fantasy",NaN,None
358,363,Scream 3,2000,waiting,movie,NaN,"comedy,horror,mystery,thriller",NaN,NaN
359,364,Gladiator,2000,waiting,movie,NaN,"action,adventure,drama",NaN,NaN
360,365,Road Trip,2000,waiting,movie,NaN,comedy,NaN,NaN
361,366,Shaft,2000,waiting,movie,NaN,"action,crime,thriller",NaN,NaN


### Recsys

In [41]:
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
def build_item_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    """Build a numeric feature matrix for item-based recommendation."""

    df_features = df.copy()
    df_features = df_features.drop(columns=['rating', 'watched_date'])

    # Min-max scaling year column
    min_year, max_year =  df['year'].min(), df['year'].max()
    df_features['year'] = (df['year'] - min_year) / (max_year - min_year)
    df_features['year'] *= 0.2  # Downweight because same year doesn't mean high similarity

    country_features = df['country'].str.get_dummies()
    country_features *= 0.4
    
    type_features = df['type'].str.get_dummies()
    type_features *= 0.5
    
    genres_features = df['genres'].str.get_dummies(',')

    return pd.concat(
        [
            df_features.drop(columns=['id', 'name', 'status', 'type', 'country', 'genres']), 
            type_features, country_features, genres_features
        ], axis=1
    )


def build_item_similarity_matrix(feature_matrix: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:
    """Compute cosine similarity between all items."""

    similarity_matrix = cosine_similarity(feature_matrix)
    return pd.DataFrame(
        similarity_matrix, 
        index=df['id'], 
        columns=df['id']
    )

In [101]:
def build_aligned_feature_matrices(personal_df, movielens_df):
    combined_df = pd.concat([personal_df, movielens_df], ignore_index=True)
    
    # Build on combined so columns are aligned
    combined_matrix = build_item_feature_matrix(combined_df)  # remove the indices, need alignment

    personal_matrix = combined_matrix.iloc[:len(personal_df)]
    movielens_matrix = combined_matrix.iloc[len(personal_df):]
    
    return personal_matrix, movielens_matrix

personal_feature_matrix, movielens_feature_matrix = \
    build_aligned_feature_matrices(personal_df, movielens_df)

# Align feature matrix indices to their respective df indices
personal_feature_matrix.index = personal_df.index
movielens_feature_matrix.index = movielens_df.index

In [97]:
def half_life(days) -> float:
    """Convert a half-life value (in days) to the exponential decay rate."""
    return np.log(2) / days

def exponential_decay(t, decay_rate: float = 0.01) -> np.ndarray:
    """Compute exponential decay weights."""
    return np.exp(-decay_rate*t)

def normalize_watched_date(date_str):
    """Normalize watched_date values into pandas Timestamp."""
    if pd.isna(date_str):
        return None
    
    date_str = str(date_str)

    # convert to mid-year for year-only and year with month data, mid-year is 1 July
    if len(date_str) == 4:  # YYYY
        return pd.Timestamp(f'{date_str}-07-01')

    if len(date_str) == 7:  # YYYY-MM
        return pd.Timestamp(f'{date_str}-15')

    return pd.Timestamp(date_str)

In [98]:
def build_user_profile(personal_df, personal_feature_matrix):
    """Build weighted user profile from personal watched movies only."""
    
    watched_df = personal_df[personal_df['status'] == 'completed']
    watched_features = personal_feature_matrix.loc[watched_df.index]
    
    rating_weights = (watched_df['rating'] / 10) ** 2
    
    watched_date = watched_df['watched_date'].apply(normalize_watched_date)
    today = pd.Timestamp.today().normalize()
    days_since_watch = (today - watched_date).dt.days
    
    decay_rate = half_life(180)
    time_decay_weights = exponential_decay(days_since_watch, decay_rate)
    
    weights = rating_weights * time_decay_weights
    return np.average(watched_features, axis=0, weights=weights)

def recommend_movielens(user_profile, movielens_df, movielens_matrix, top_k=10):
    """Recommend from MovieLens candidate pool using personal user profile."""

    scores = cosine_similarity([user_profile], movielens_matrix).flatten()
    sim_series = pd.Series(scores, index=movielens_df.index)

    top_idx = sim_series.sort_values(ascending=False).head(top_k).index
    results = (
        movielens_df.loc[top_idx]
        .reset_index(drop=True)
        .drop(columns=['status', 'type', 'country', 'rating', 'watched_date'])
    )
    return results

In [99]:
user_profile = build_user_profile(personal_df, personal_feature_matrix)
user_profile.shape

(47,)

In [102]:
recommend_movielens(
    user_profile, movielens_df, movielens_feature_matrix, top_k=10
)

,id,name,year,genres
0,4305,"Librarian: Quest for the Spear, The",2004,"action,adventure,comedy,fantasy,romance"
1,3458,Mind Game,2004,"adventure,animation,comedy,fantasy,romance,sci-fi"
2,1647,Futurama: Bender's Game,2008,"action,adventure,animation,comedy,fantasy,sci-fi"
3,2556,Futurama: The Beast with a Billion Backs,2008,"action,animation,comedy,romance,sci-fi"
4,1632,Fool's Gold,2008,"action,adventure,comedy,romance"
5,535,Mr. & Mrs. Smith,2005,"action,adventure,comedy,romance"
6,3630,Aqua Teen Hunger Force Colon Movie Film for Th...,2007,"action,adventure,animation,comedy,fantasy,myst..."
7,591,Frozen,2013,"adventure,animation,comedy,fantasy,music,romance"
8,1915,The Amazing Screw-On Head,2006,"action,adventure,animation,comedy,sci-fi"
9,696,Team America: World Police,2004,"action,adventure,animation,comedy"


In [103]:
recommend_movielens(
    user_profile, movielens_df, movielens_feature_matrix, top_k=20
)

,id,name,year,genres
0,4305,"Librarian: Quest for the Spear, The",2004,"action,adventure,comedy,fantasy,romance"
1,3458,Mind Game,2004,"adventure,animation,comedy,fantasy,romance,sci-fi"
2,1647,Futurama: Bender's Game,2008,"action,adventure,animation,comedy,fantasy,sci-fi"
3,2556,Futurama: The Beast with a Billion Backs,2008,"action,animation,comedy,romance,sci-fi"
4,1632,Fool's Gold,2008,"action,adventure,comedy,romance"
5,535,Mr. & Mrs. Smith,2005,"action,adventure,comedy,romance"
6,3630,Aqua Teen Hunger Force Colon Movie Film for Th...,2007,"action,adventure,animation,comedy,fantasy,myst..."
7,591,Frozen,2013,"adventure,animation,comedy,fantasy,music,romance"
8,1915,The Amazing Screw-On Head,2006,"action,adventure,animation,comedy,sci-fi"
9,696,Team America: World Police,2004,"action,adventure,animation,comedy"


### Blacklist

In [120]:
import json
from pathlib import Path

BLACKLIST_PATH = Path('blacklist.json')

def load_blacklist() -> set:
    if not BLACKLIST_PATH.exists():
        return set()
    with open(BLACKLIST_PATH) as f:
        return set(json.load(f)['blacklisted_ids'])

def save_blacklist(blacklist: set):
    BLACKLIST_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(BLACKLIST_PATH, 'w') as f:
        json.dump({'blacklisted_ids': list(blacklist)}, f, separators=(',', ':'))

def add_to_blacklist(movie_ids: list):
    blacklist = load_blacklist()
    blacklist.update(movie_ids)
    save_blacklist(blacklist)

def filter_blacklist(df: pd.DataFrame) -> pd.DataFrame:
    blacklist = load_blacklist()
    return df[~df['id'].isin(blacklist)]

In [111]:
def recommend_filtered_movielens() -> pd.DataFrame:
    clean_movielens_df = filter_blacklist(movielens_df)
    return recommend_movielens(
        user_profile,
        clean_movielens_df,
        movielens_feature_matrix.loc[clean_movielens_df.index],  # align matrix
        top_k=10
    )

results = recommend_filtered_movielens()
results

,id,name,year,genres
0,4305,"Librarian: Quest for the Spear, The",2004,"action,adventure,comedy,fantasy,romance"
1,3458,Mind Game,2004,"adventure,animation,comedy,fantasy,romance,sci-fi"
2,1632,Fool's Gold,2008,"action,adventure,comedy,romance"
3,535,Mr. & Mrs. Smith,2005,"action,adventure,comedy,romance"
4,3630,Aqua Teen Hunger Force Colon Movie Film for Th...,2007,"action,adventure,animation,comedy,fantasy,myst..."
5,591,Frozen,2013,"adventure,animation,comedy,fantasy,music,romance"
6,1915,The Amazing Screw-On Head,2006,"action,adventure,animation,comedy,sci-fi"
7,696,Team America: World Police,2004,"action,adventure,animation,comedy"
8,2549,Casanova,2005,"action,adventure,comedy,drama,romance"
9,1427,Lara Croft Tomb Raider: The Cradle of Life,2003,"action,adventure,comedy,romance,thriller"


In [113]:
results['id'].tolist()

[4305, 3458, 1632, 535, 3630, 591, 1915, 696, 2549, 1427]

In [114]:
add_to_blacklist(results['id'].tolist())

In [117]:
results = recommend_filtered_movielens()
results

,id,name,year,genres
0,2703,Black Butler: Book of the Atlantic,2017,"action,animation,comedy,fantasy"
1,2374,Journey to the West: Conquering the Demons (Da...,2013,"adventure,comedy,fantasy,romance"
2,2243,Mr. Right,2016,"action,comedy,romance"
3,3936,Charlie Countryman,2013,"action,comedy,romance"
4,1341,This Means War,2012,"action,comedy,romance"
5,1488,Stardust,2007,"adventure,comedy,fantasy,romance"
6,1705,"Bounty Hunter, The",2010,"action,comedy,romance"
7,942,Knight and Day,2010,"action,comedy,romance"
8,1706,Date Night,2010,"action,comedy,romance"
9,774,Scott Pilgrim vs. the World,2010,"action,comedy,fantasy,music,romance"


In [119]:
add_to_blacklist(results['id'].tolist()[1:])
results = recommend_filtered_movielens()
results

,id,name,year,genres
0,2703,Black Butler: Book of the Atlantic,2017,"action,animation,comedy,fantasy"
1,1391,Ant-Man and the Wasp,2018,"action,adventure,comedy,fantasy,sci-fi"
2,875,"Knight's Tale, A",2001,"action,comedy,romance"
3,2191,Your Highness,2011,"action,adventure,comedy,fantasy"
4,2424,"Forbidden Kingdom, The",2008,"action,adventure,comedy,fantasy"
5,1838,"Gamers, The: Dorkness Rising",2008,"action,adventure,comedy,fantasy"
6,723,Pirates of the Caribbean: At World's End,2007,"action,adventure,comedy,fantasy"
7,451,Pirates of the Caribbean: The Curse of the Bla...,2003,"action,adventure,comedy,fantasy"
8,1224,Dungeons & Dragons,2000,"action,adventure,comedy,fantasy"
9,1442,Prince of Persia: The Sands of Time,2010,"action,adventure,fantasy,romance"


In [121]:
add_to_blacklist(results['id'].tolist()[2:])
results = recommend_filtered_movielens()
results

,id,name,year,genres
0,2703,Black Butler: Book of the Atlantic,2017,"action,animation,comedy,fantasy"
1,1391,Ant-Man and the Wasp,2018,"action,adventure,comedy,fantasy,sci-fi"
2,3552,Teenage Mutant Ninja Turtles: Turtles Forever,2009,"action,adventure,animation,comedy,thriller"
3,708,Corpse Bride,2005,"animation,comedy,fantasy,music,romance"
4,1497,Paperman,2012,"animation,comedy,romance"
5,2567,Puss in Boots,2011,"adventure,animation,comedy,fantasy"
6,3549,Shrek the Halls,2007,"adventure,animation,comedy,fantasy"
7,2688,Sword Art Online The Movie: Ordinal Scale,2017,"action,adventure,animation,fantasy,sci-fi"
8,2265,The Lego Batman Movie,2017,"action,animation,comedy"
9,3551,Green Lantern: First Flight,2009,"action,adventure,animation,fantasy,sci-fi"
